In [55]:
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt
import kagglehub
from kagglehub import KaggleDatasetAdapter
import os.path
from sklearn.model_selection import train_test_split as split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, fbeta_score, precision_score, recall_score, roc_auc_score, make_scorer
from sklearn.model_selection import GridSearchCV

In [56]:
df = pd.read_csv("../data/processed/f1_strategy_dataset_v4_processed.csv")

In [57]:
X = df.drop('PitNextLap', axis=1)
y = df['PitNextLap']

X_train, X_test, y_train, y_test = split(X, y, test_size=0.2, random_state=42)
print(f"Доля пит-стопов в трейне: {y_train.mean():.4f}")
print(f"Доля пит-стопов в тесте: {y_test.mean():.4f}")

Доля пит-стопов в трейне: 0.2539
Доля пит-стопов в тесте: 0.2583


In [58]:
scaler = StandardScaler()
scaler.fit(X_train)

X_test_scaled = scaler.transform(X_test)
X_train_scaled = scaler.transform(X_train)

X_train = X_train_scaled
X_test = X_test_scaled

In [59]:
def print_metrics(model_name, y_true, y_pred):    
    f2 = fbeta_score(y_true, y_pred, beta=2)
    acc = accuracy_score(y_true, y_pred)
    precision = precision_score(y_test, y_pred, average='macro')
    recall = recall_score(y_true, y_pred, average='macro')
    
    print("="*50)
    print(f"Model name: {model_name}")
    print(f"F2-score: {f2:.2f}")
    print(f"Accuracy: {acc:.2f}")
    print(f"Precision: {precision:.2f}")
    print(f"Recall: {recall:.2f}")
    print("="*50)

In [60]:
f2_scorer = make_scorer(fbeta_score, beta=2)

In [61]:
def train_grid_search(model_name, trainer, X_train, y_train, X_test, y_test):
    gs = trainer(X_train, y_train)
    print(f'best CV F1-macro: {gs.best_score_:.4f}, params: {gs.best_params_}')
    best_model = gs.best_estimator_
    y_pred = best_model.predict(X_test)
    print_metrics(model_name, y_test, y_pred)
    return gs.best_estimator_

In [62]:
def train_logreg(X_train, y_train):
    model = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')

    param_grid = {
        'C': [0.01, 0.1, 1, 10, 100],
        'solver': ['liblinear', 'saga'],
        'penalty': ['l1', 'l2']
    }

    grid_search = GridSearchCV(
        estimator=model,
        param_grid=param_grid,
        scoring=f2_scorer,
        cv=5,
        n_jobs=-1,
        verbose=1,
        return_train_score=True
    )

    grid_search.fit(X_train, y_train)

    return grid_search

In [63]:
def train_svm(X_train, y_train) -> GridSearchCV:
    model = SVC(max_iter=1000, random_state=42, kernel='linear', class_weight='balanced', probability=True)

    param_grid = {
        'C': [0.01, 0.1, 1, 10, 100]
    }
    

    grid_search = GridSearchCV(
        estimator=model,
        param_grid=param_grid,
        scoring=f2_scorer,
        cv=5,
        n_jobs=-1,
        verbose=1,
        return_train_score=True
    )

    grid_search.fit(X_train, y_train)

    return grid_search

In [64]:
def train_random_forest(X_train, y_train):
    model = RandomForestClassifier(random_state=42, class_weight='balanced_subsample')

    param_grid = {
        'n_estimators': [1, 5, 10, 50, 100],
        'max_depth': [10, 25, 100, None],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4] 
    }

    grid_search = GridSearchCV(
        estimator=model,
        param_grid=param_grid,
        scoring=f2_scorer,
        cv=5,
        n_jobs=-1,
        verbose=1,
        return_train_score=True
    )

    grid_search.fit(X_train, y_train)

    return grid_search

In [65]:
logreg = LogisticRegression()
logreg.fit(X_train_scaled, y_train)
y_pred = logreg.predict(X_test_scaled)
print_metrics('Base logreg', y_test, y_pred)

best_logreg = train_grid_search('Logistic regression', train_logreg, X_train_scaled, y_train, X_test, y_test)

best_svm = train_grid_search('SVM', train_svm, X_train_scaled, y_train, X_test, y_test)

best_random_forest = train_grid_search('Random forest', train_random_forest, X_train_scaled, y_train, X_test, y_test)

Model name: Base logreg
F2-score: 0.33
Accuracy: 0.77
Precision: 0.70
Recall: 0.62
Fitting 5 folds for each of 20 candidates, totalling 100 fits


D:\PyCharm\hseml-group-project-vassuha\hseml-group-project\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
D:\PyCharm\hseml-group-project-vassuha\hseml-group-project\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


best CV F1-macro: 0.6350, params: {'C': 0.01, 'penalty': 'l1', 'solver': 'liblinear'}
Model name: Logistic regression
F2-score: 0.63
Accuracy: 0.70
Precision: 0.66
Recall: 0.70
Fitting 5 folds for each of 5 candidates, totalling 25 fits


D:\PyCharm\hseml-group-project-vassuha\hseml-group-project\Lib\site-packages\sklearn\svm\_base.py:313: ConvergenceWarning: Solver terminated early (max_iter=1000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(


best CV F1-macro: 0.6298, params: {'C': 0.01}
Model name: SVM
F2-score: 0.64
Accuracy: 0.26
Precision: 0.63
Recall: 0.50
Fitting 5 folds for each of 180 candidates, totalling 900 fits
best CV F1-macro: 0.9285, params: {'max_depth': 100, 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 100}
Model name: Random forest
F2-score: 0.94
Accuracy: 0.97
Precision: 0.97
Recall: 0.96
